# Global Hashtag Association Mining


# Hashtag Association Mining

This notebook treats each post as one transaction and each hashtag as an item.


In [1]:

import re
from pathlib import Path

import numpy as np
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 180)


In [2]:

DATA_PATH = Path('synthetic_generator') / 'synthetic_social_media_posts_with_kpis.csv'
OUTPUT_DIR = Path('notebooks') / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
print('Rows:', len(df))
print('Columns:', list(df.columns))


Rows: 1000
Columns: ['business_name', 'sector', 'followers_count', 'post_date', 'posting_hour', 'day_of_week', 'month', 'post_type', 'caption_text', 'caption_length', 'hashtags_count', 'emoji_count', 'likes_count', 'comments_count', 'views_count', 'language', 'CTA_present', 'promo_post', 'discount_percent', 'mentions_location', 'religious_theme', 'patriotic_theme', 'arabic_dialect_style', 'engagement_count', 'engagement_rate_by_followers', 'like_rate_by_views', 'comment_rate_by_likes', 'views_per_follower', 'is_high_engagement', 'posting_time_category', 'caption_length_category', 'hashtag_intensity', 'content_localization_score']


In [3]:

# Regex captures hashtags in Arabic/English, digits, and underscore.
HASHTAG_RE = re.compile(r'(?<!\w)#([\w\u0600-\u06FF]+)', flags=re.UNICODE)

def extract_hashtags(text):
    if pd.isna(text):
        return []
    tags = [f"#{m.lower()}" for m in HASHTAG_RE.findall(str(text))]
    # de-duplicate within each post transaction
    return sorted(set(tags))

df['hashtags'] = df['caption_text'].apply(extract_hashtags)
df['n_tags_extracted'] = df['hashtags'].str.len()

print('Posts with >=1 hashtag:', int((df['n_tags_extracted'] > 0).sum()))
print('Unique hashtags:', len(set(t for tags in df['hashtags'] for t in tags)))

df[['caption_text', 'hashtags']].head(5)


Posts with >=1 hashtag: 680
Unique hashtags: 16


,caption_text,hashtags
0,صحتك بتهمنا Swipe وشوفوا الباقي. موجودين في Rafah today. منتج فلسطيني زورونا وخلينا نجهزلك طلبك. #تعليم #مخبز,"[#تعليم, #مخبز]"
1,اليوم عنا أكل طيب فيديو جديد. احنا جاهزين احجز الآن وخليها علينا #عيادة #تعليم,"[#تعليم, #عيادة]"
2,صحتكم أولويتنا سوايب وشوفوا التفاصيل. شو رأيكم؟ الكمية محدودة 50% خصم لفترة محدودة زورونا بفرعنا في Rafah ابعتولنا D...,[#عروض]
3,استنونا بدورة جديدة شوفوا الريل. زورونا بفرعنا في Tulkarm كل عام وأنتم بخير ✅ #تعليم #قهوة,"[#تعليم, #قهوة]"
4,أجواء ولا أروع صورة جديدة. أهلا بالناس الحلوة منتج فلسطيني #عيادة #مطاعم,"[#عيادة, #مطاعم]"


In [4]:

transactions = df.loc[df['n_tags_extracted'] > 0, 'hashtags'].tolist()

te = TransactionEncoder()
te_arr = te.fit(transactions).transform(transactions)
onehot = pd.DataFrame(te_arr, columns=te.columns_)

# Moderate threshold to keep meaningful global pairs.
freq = apriori(onehot, min_support=0.01, use_colnames=True)
freq = freq.sort_values(['support', 'itemsets'], ascending=[False, True]).reset_index(drop=True)

rules = association_rules(freq, metric='confidence', min_threshold=0.2)
rules = rules.sort_values(['lift', 'confidence', 'support'], ascending=[False, False, False]).reset_index(drop=True)

for col in ['antecedents', 'consequents']:
    rules[col] = rules[col].apply(lambda x: ', '.join(sorted(list(x))))

print('Frequent itemsets:', len(freq))
print('Association rules:', len(rules))

best_global = rules.head(20).copy()
best_global.to_csv(OUTPUT_DIR / 'global_hashtag_best_rules.csv', index=False)

best_global[['antecedents','consequents','support','confidence','lift']]


Frequent itemsets: 100
Association rules: 51


,antecedents,consequents,support,confidence,lift
0,#nablus,#عروض,0.016176,0.458333,1.712454
1,"#palestine, #مطاعم",#تسوق,0.011765,0.222222,1.624851
2,#ramallah,#مطاعم,0.017647,0.300000,1.457143
3,#nablus,#مطاعم,0.010294,0.291667,1.416667
4,"#تسوق, #عروض",#palestine,0.016176,0.478261,1.360742
5,#الكترونيات,#تعليم,0.023529,0.200000,1.333333
6,#hebron,#عروض,0.017647,0.333333,1.245421
7,"#palestine, #عيادة",#عروض,0.016176,0.323529,1.208791
8,"#palestine, #تسوق",#عروض,0.016176,0.323529,1.208791
9,"#عيادة, #قهوة",#palestine,0.010294,0.411765,1.171548


In [5]:

if len(rules) == 0:
    summary = pd.DataFrame([{
        'scope': 'global',
        'note': 'No rules found with current thresholds',
        'n_transactions': len(transactions),
        'n_unique_hashtags': onehot.shape[1]
    }])
else:
    summary = pd.DataFrame([{
        'scope': 'global',
        'n_transactions': len(transactions),
        'n_unique_hashtags': onehot.shape[1],
        'n_rules': len(rules),
        'best_antecedent': best_global.iloc[0]['antecedents'],
        'best_consequent': best_global.iloc[0]['consequents'],
        'best_support': float(best_global.iloc[0]['support']),
        'best_confidence': float(best_global.iloc[0]['confidence']),
        'best_lift': float(best_global.iloc[0]['lift'])
    }])

summary.to_csv(OUTPUT_DIR / 'global_hashtag_summary.csv', index=False)
summary


,scope,n_transactions,n_unique_hashtags,n_rules,best_antecedent,best_consequent,best_support,best_confidence,best_lift
0,global,680,16,51,#nablus,#عروض,0.016176,0.458333,1.712454
